In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

In [2]:
merged = pd.read_csv(
    processed_path / "merged_raw.csv"
)

merged["timestamp"] = pd.to_datetime(
    merged["timestamp"],
    errors="coerce"
)

print(merged.shape)
print(merged.columns.tolist())

(65535, 26)
['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose']


In [3]:
merged = merged.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

In [4]:
print(merged[[
    "Patient",
    "timestamp",
    "glucose_value"
]].head(10))

           Patient           timestamp  glucose_value
0  540-ws-training 2027-05-19 11:36:29             76
1  540-ws-training 2027-05-19 11:41:29             72
2  540-ws-training 2027-05-19 11:46:29             68
3  540-ws-training 2027-05-19 11:51:29             65
4  540-ws-training 2027-05-19 11:56:29             63
5  540-ws-training 2027-05-19 12:01:29             66
6  540-ws-training 2027-05-19 12:06:29             71
7  540-ws-training 2027-05-19 12:11:29             78
8  540-ws-training 2027-05-19 12:16:29             90
9  540-ws-training 2027-05-19 12:21:29             99


In [5]:
merged["hour"] = merged["timestamp"].dt.hour

merged["day_of_week"] = merged["timestamp"].dt.dayofweek

merged["is_weekend"] = (
    merged["day_of_week"] >= 5
).astype(int)

In [6]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

merged["time_period"] = merged["hour"].apply(get_time_period)

In [7]:
print(
    merged[[
        "timestamp",
        "hour",
        "day_of_week",
        "is_weekend",
        "time_period"
    ]].head(20)
)

             timestamp  hour  day_of_week  is_weekend time_period
0  2027-05-19 11:36:29    11            2           0     morning
1  2027-05-19 11:41:29    11            2           0     morning
2  2027-05-19 11:46:29    11            2           0     morning
3  2027-05-19 11:51:29    11            2           0     morning
4  2027-05-19 11:56:29    11            2           0     morning
5  2027-05-19 12:01:29    12            2           0   afternoon
6  2027-05-19 12:06:29    12            2           0   afternoon
7  2027-05-19 12:11:29    12            2           0   afternoon
8  2027-05-19 12:16:29    12            2           0   afternoon
9  2027-05-19 12:21:29    12            2           0   afternoon
10 2027-05-19 12:26:29    12            2           0   afternoon
11 2027-05-19 12:31:29    12            2           0   afternoon
12 2027-05-19 12:36:29    12            2           0   afternoon
13 2027-05-19 12:41:29    12            2           0   afternoon
14 2027-05

In [8]:
print(merged["time_period"].value_counts())

time_period
night        22310
morning      18479
afternoon    13602
evening      11144
Name: count, dtype: int64


In [9]:
merged = merged.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

In [10]:
merged["glucose_prev_5min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(1)
)

merged["glucose_prev_10min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(2)
)

merged["glucose_prev_15min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(3)
)

merged["glucose_prev_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(6)
)

In [11]:
merged["glucose_change_5min"] = (
    merged["glucose_value"] - merged["glucose_prev_5min"]
)

merged["glucose_change_10min"] = (
    merged["glucose_value"] - merged["glucose_prev_10min"]
)

merged["glucose_change_15min"] = (
    merged["glucose_value"] - merged["glucose_prev_15min"]
)

merged["glucose_change_30min"] = (
    merged["glucose_value"] - merged["glucose_prev_30min"]
)

merged["glucose_rate_15min"] = (
    merged["glucose_change_15min"] / 15
)

merged["glucose_rate_30min"] = (
    merged["glucose_change_30min"] / 30
)

In [12]:
merged["glucose_rate_5min"] = (
    merged["glucose_change_5min"] / 5
)

merged["glucose_rate_15min"] = (
    merged["glucose_change_15min"] / 15
)

merged["glucose_rate_30min"] = (
    merged["glucose_change_30min"] / 30
)

In [13]:
print(
    merged[[
        "Patient",
        "timestamp",
        "glucose_value",
        "glucose_change_5min",
        "glucose_change_15min",
        "glucose_change_30min",
        "glucose_rate_5min",
        "glucose_rate_15min",
        "glucose_rate_30min"
    ]].head(15)
)

            Patient           timestamp  glucose_value  glucose_change_5min  \
0   540-ws-training 2027-05-19 11:36:29             76                  NaN   
1   540-ws-training 2027-05-19 11:41:29             72                 -4.0   
2   540-ws-training 2027-05-19 11:46:29             68                 -4.0   
3   540-ws-training 2027-05-19 11:51:29             65                 -3.0   
4   540-ws-training 2027-05-19 11:56:29             63                 -2.0   
5   540-ws-training 2027-05-19 12:01:29             66                  3.0   
6   540-ws-training 2027-05-19 12:06:29             71                  5.0   
7   540-ws-training 2027-05-19 12:11:29             78                  7.0   
8   540-ws-training 2027-05-19 12:16:29             90                 12.0   
9   540-ws-training 2027-05-19 12:21:29             99                  9.0   
10  540-ws-training 2027-05-19 12:26:29            110                 11.0   
11  540-ws-training 2027-05-19 12:31:29            1

In [14]:
print(
    merged[[
        "glucose_change_5min",
        "glucose_change_15min",
        "glucose_change_30min"
    ]].describe()
)

       glucose_change_5min  glucose_change_15min  glucose_change_30min
count         65529.000000          65517.000000          65499.000000
mean              0.011461              0.035136              0.069925
std               7.201349             16.482178             27.225224
min            -286.000000           -301.000000           -317.000000
25%              -3.000000             -8.000000            -14.000000
50%               0.000000             -1.000000             -1.000000
75%               3.000000              7.000000             13.000000
max             297.000000            310.000000            310.000000


Step 3 — Rolling Glucose Statistics

In [15]:
merged["glucose_rolling_mean_15min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

In [16]:
merged["glucose_rolling_mean_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

In [17]:
merged["glucose_rolling_std_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=2).std())
)

In [18]:
merged["glucose_min_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).min())
)

merged["glucose_max_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).max())
)

In [19]:
print(
    merged[[
        "Patient",
        "timestamp",
        "glucose_value",
        "glucose_rolling_mean_15min",
        "glucose_rolling_mean_30min",
        "glucose_rolling_std_30min",
        "glucose_min_30min",
        "glucose_max_30min"
    ]].head(15)
)

            Patient           timestamp  glucose_value  \
0   540-ws-training 2027-05-19 11:36:29             76   
1   540-ws-training 2027-05-19 11:41:29             72   
2   540-ws-training 2027-05-19 11:46:29             68   
3   540-ws-training 2027-05-19 11:51:29             65   
4   540-ws-training 2027-05-19 11:56:29             63   
5   540-ws-training 2027-05-19 12:01:29             66   
6   540-ws-training 2027-05-19 12:06:29             71   
7   540-ws-training 2027-05-19 12:11:29             78   
8   540-ws-training 2027-05-19 12:16:29             90   
9   540-ws-training 2027-05-19 12:21:29             99   
10  540-ws-training 2027-05-19 12:26:29            110   
11  540-ws-training 2027-05-19 12:31:29            121   
12  540-ws-training 2027-05-19 12:36:29            131   
13  540-ws-training 2027-05-19 12:41:29            137   
14  540-ws-training 2027-05-19 12:46:29            140   

    glucose_rolling_mean_15min  glucose_rolling_mean_30min  \
0        

In [20]:
print(
    merged[[
        "glucose_rolling_mean_15min",
        "glucose_rolling_mean_30min",
        "glucose_rolling_std_30min",
        "glucose_min_30min",
        "glucose_max_30min"
    ]].describe()
)

       glucose_rolling_mean_15min  glucose_rolling_mean_30min  \
count                65535.000000                65535.000000   
mean                   157.688739                  157.671140   
std                     60.650731                   60.175888   
min                     40.000000                   40.000000   
25%                    112.333333                  112.666667   
50%                    149.333333                  149.500000   
75%                    193.000000                  192.666667   
max                    400.000000                  400.000000   

       glucose_rolling_std_30min  glucose_min_30min  glucose_max_30min  
count               65529.000000       65535.000000       65535.000000  
mean                    7.080267         148.620294         166.764141  
std                     7.220368          58.662400          62.190581  
min                     0.000000          40.000000          40.000000  
25%                     2.732520         105.0000

Insulin Features

In [21]:
merged["bolus_given"] = (
    merged["bolus_dose"].notna()
).astype(int)

In [22]:
print(merged["bolus_given"].value_counts())

bolus_given
1    65294
0      241
Name: count, dtype: int64


In [23]:
merged["bolus_dose"] = pd.to_numeric(
    merged["bolus_dose"],
    errors="coerce"
)

merged["bolus_dose_filled"] = (
    merged["bolus_dose"].fillna(0)
)

In [24]:
merged["bolus_30min"] = (
    merged.groupby("Patient")["bolus_dose_filled"]
    .transform(
        lambda x: x.rolling(6, min_periods=1).sum()
    )
)

In [25]:
merged["bolus_60min"] = (
    merged.groupby("Patient")["bolus_dose_filled"]
    .transform(
        lambda x: x.rolling(12, min_periods=1).sum()
    )
)

In [26]:
print(
    merged[[
        "Patient",
        "timestamp",
        "bolus_dose",
        "bolus_given",
        "bolus_30min",
        "bolus_60min"
    ]].head(20)
)

            Patient           timestamp  bolus_dose  bolus_given  bolus_30min  \
0   540-ws-training 2027-05-19 11:36:29         0.8            1          0.8   
1   540-ws-training 2027-05-19 11:41:29         0.8            1          1.6   
2   540-ws-training 2027-05-19 11:46:29         0.8            1          2.4   
3   540-ws-training 2027-05-19 11:51:29         0.8            1          3.2   
4   540-ws-training 2027-05-19 11:56:29         0.8            1          4.0   
5   540-ws-training 2027-05-19 12:01:29         0.8            1          4.8   
6   540-ws-training 2027-05-19 12:06:29         0.8            1          4.8   
7   540-ws-training 2027-05-19 12:11:29         5.5            1          9.5   
8   540-ws-training 2027-05-19 12:16:29         5.5            1         14.2   
9   540-ws-training 2027-05-19 12:21:29         5.5            1         18.9   
10  540-ws-training 2027-05-19 12:26:29         5.5            1         23.6   
11  540-ws-training 2027-05-

In [27]:
print(
    merged[[
        "bolus_dose",
        "bolus_30min",
        "bolus_60min"
    ]].describe()
)

         bolus_dose   bolus_30min   bolus_60min
count  65294.000000  65535.000000  65535.000000
mean       6.792060     40.592538     81.162515
std        4.854745     28.774137     56.727741
min        0.100000      0.000000      0.000000
25%        3.000000     18.000000     36.000000
50%        5.900000     35.400000     70.800000
75%        9.300000     55.800000    111.600000
max       25.000000    150.000000    300.000000


In [28]:
print(merged["basal_value"].describe())

count    65535.000000
mean         1.067877
std          0.510238
min          0.400000
25%          0.450000
50%          1.100000
75%          1.500000
max          2.000000
Name: basal_value, dtype: float64


In [29]:
merged["basal_rolling_30min"] = (
    merged.groupby("Patient")["basal_value"]
    .transform(
        lambda x: x.rolling(6, min_periods=1).mean()
    )
)

In [30]:
merged["basal_rolling_60min"] = (
    merged.groupby("Patient")["basal_value"]
    .transform(
        lambda x: x.rolling(12, min_periods=1).mean()
    )
)

In [31]:
merged["temp_basal_active"] = (
    merged["temp_basal_value"].notna()
).astype(int)

In [32]:
merged["temp_basal_value"] = pd.to_numeric(
    merged["temp_basal_value"],
    errors="coerce"
)

In [33]:
merged["effective_basal"] = merged["basal_value"]

In [34]:
merged.loc[
    merged["temp_basal_value"].notna(),
    "effective_basal"
] = merged.loc[
    merged["temp_basal_value"].notna(),
    "temp_basal_value"
]

In [35]:
merged["basal_change"] = (
    merged["effective_basal"] -
    merged["basal_value"]
)

In [36]:
print(
    merged[[
        "timestamp",
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "temp_basal_active",
        "basal_rolling_30min",
        "basal_rolling_60min"
    ]].head(20)
)

             timestamp  basal_value  temp_basal_value  effective_basal  \
0  2027-05-19 11:36:29         0.95               NaN             0.95   
1  2027-05-19 11:41:29         0.95               NaN             0.95   
2  2027-05-19 11:46:29         0.95               NaN             0.95   
3  2027-05-19 11:51:29         0.95               NaN             0.95   
4  2027-05-19 11:56:29         0.95               NaN             0.95   
5  2027-05-19 12:01:29         0.95               NaN             0.95   
6  2027-05-19 12:06:29         0.95               NaN             0.95   
7  2027-05-19 12:11:29         0.95               NaN             0.95   
8  2027-05-19 12:16:29         0.95               NaN             0.95   
9  2027-05-19 12:21:29         0.95               NaN             0.95   
10 2027-05-19 12:26:29         0.95               NaN             0.95   
11 2027-05-19 12:31:29         0.95               NaN             0.95   
12 2027-05-19 12:36:29         0.95   

In [37]:
print(
    merged[[
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "basal_rolling_30min",
        "basal_rolling_60min"
    ]].describe()
)

        basal_value  temp_basal_value  effective_basal  basal_change  \
count  65535.000000      58609.000000     65535.000000  65535.000000   
mean       1.067877          0.139367         0.200766     -0.867111   
std        0.510238          0.403941         0.444413      0.775335   
min        0.400000          0.000000         0.000000     -2.000000   
25%        0.450000          0.000000         0.000000     -1.500000   
50%        1.100000          0.000000         0.000000     -1.000000   
75%        1.500000          0.000000         0.200000     -0.250000   
max        2.000000          2.340000         2.340000      1.700000   

       basal_rolling_30min  basal_rolling_60min  
count         65535.000000         65535.000000  
mean              1.067908             1.067944  
std               0.509748             0.509250  
min               0.400000             0.400000  
25%               0.450000             0.450000  
50%               1.100000             1.100000  
7

In [38]:
merged["temp_basal_active"] = (
    merged["temp_basal_value"] > 0
).astype(int)

In [39]:
merged["effective_basal"] = merged["basal_value"]

mask = merged["temp_basal_value"] > 0

merged.loc[mask, "effective_basal"] = (
    merged.loc[mask, "temp_basal_value"]
)

In [40]:
merged["basal_change"] = (
    merged["effective_basal"] -
    merged["basal_value"]
)

In [41]:
print(
    merged[[
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "temp_basal_active"
    ]].describe()
)

        basal_value  temp_basal_value  effective_basal  basal_change  \
count  65535.000000      58609.000000     65535.000000  65535.000000   
mean       1.067877          0.139367         1.105268      0.037390   
std        0.510238          0.403941         0.540113      0.275848   
min        0.400000          0.000000         0.020000     -0.600000   
25%        0.450000          0.000000         0.450000      0.000000   
50%        1.100000          0.000000         1.100000      0.000000   
75%        1.500000          0.000000         1.600000      0.000000   
max        2.000000          2.340000         2.340000      1.700000   

       temp_basal_active  
count       65535.000000  
mean            0.179980  
std             0.384174  
min             0.000000  
25%             0.000000  
50%             0.000000  
75%             0.000000  
max             1.000000  


Meal & Carbohydrate Features

In [42]:
merged["meal_carbs"] = pd.to_numeric(
    merged["meal_carbs"],
    errors="coerce"
)

In [43]:
merged["meal_event"] = (
    merged["meal_carbs"].notna() &
    (merged["meal_carbs"] > 0)
).astype(int)

In [44]:
merged["carbs_30min"] = (
    merged.groupby("Patient")["meal_carbs"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(6, min_periods=1)
        .sum()
    )
)

In [45]:
merged["carbs_60min"] = (
    merged.groupby("Patient")["meal_carbs"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(12, min_periods=1)
        .sum()
    )
)

In [46]:
print(
    merged[[
        "timestamp",
        "meal_type",
        "meal_carbs",
        "meal_event",
        "carbs_30min",
        "carbs_60min"
    ]].head(30)
)

             timestamp meal_type  meal_carbs  meal_event  carbs_30min  \
0  2027-05-19 11:36:29       NaN         NaN           0          0.0   
1  2027-05-19 11:41:29       NaN         NaN           0          0.0   
2  2027-05-19 11:46:29       NaN         NaN           0          0.0   
3  2027-05-19 11:51:29       NaN         NaN           0          0.0   
4  2027-05-19 11:56:29       NaN         NaN           0          0.0   
5  2027-05-19 12:01:29       NaN         NaN           0          0.0   
6  2027-05-19 12:06:29       NaN         NaN           0          0.0   
7  2027-05-19 12:11:29       NaN         NaN           0          0.0   
8  2027-05-19 12:16:29       NaN         NaN           0          0.0   
9  2027-05-19 12:21:29       NaN         NaN           0          0.0   
10 2027-05-19 12:26:29       NaN         NaN           0          0.0   
11 2027-05-19 12:31:29       NaN         NaN           0          0.0   
12 2027-05-19 12:36:29       NaN         NaN       

In [47]:
print(
    merged[[
        "meal_carbs",
        "carbs_30min",
        "carbs_60min"
    ]].describe()
)

        meal_carbs   carbs_30min   carbs_60min
count  64026.00000  65535.000000  65535.000000
mean      55.84650    327.298054    654.439002
std       30.47923    186.376428    370.187733
min        1.00000      0.000000      0.000000
25%       30.00000    180.000000    360.000000
50%       60.00000    360.000000    720.000000
75%       75.00000    434.500000    864.000000
max      162.00000    972.000000   1944.000000


In [48]:
merged["exercise_event"] = (
    merged["exercise_duration"].notna() &
    (merged["exercise_duration"] > 0)
).astype(int)

In [49]:
merged["exercise_duration"] = pd.to_numeric(
    merged["exercise_duration"],
    errors="coerce"
)

In [50]:
merged["exercise_duration_30min"] = (
    merged.groupby("Patient")["exercise_duration"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(6, min_periods=1)
        .sum()
    )
)

In [51]:
merged["exercise_duration_60min"] = (
    merged.groupby("Patient")["exercise_duration"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(12, min_periods=1)
        .sum()
    )
)

In [52]:
print(merged["exercise_intensity"].value_counts(dropna=False))

exercise_intensity
NaN     35807
10.0    10146
5.0      5549
6.0      5127
3.0      3259
7.0      2737
8.0      1780
9.0       847
4.0       283
Name: count, dtype: int64


In [53]:
print(
    merged[[
        "timestamp",
        "exercise_type",
        "exercise_duration",
        "exercise_intensity",
        "exercise_event",
        "exercise_duration_30min",
        "exercise_duration_60min"
    ]].head(30)
)

             timestamp exercise_type  exercise_duration  exercise_intensity  \
0  2027-05-19 11:36:29           NaN                NaN                 NaN   
1  2027-05-19 11:41:29           NaN                NaN                 NaN   
2  2027-05-19 11:46:29           NaN                NaN                 NaN   
3  2027-05-19 11:51:29           NaN                NaN                 NaN   
4  2027-05-19 11:56:29           NaN                NaN                 NaN   
5  2027-05-19 12:01:29           NaN                NaN                 NaN   
6  2027-05-19 12:06:29           NaN                NaN                 NaN   
7  2027-05-19 12:11:29           NaN                NaN                 NaN   
8  2027-05-19 12:16:29           NaN                NaN                 NaN   
9  2027-05-19 12:21:29           NaN                NaN                 NaN   
10 2027-05-19 12:26:29           NaN                NaN                 NaN   
11 2027-05-19 12:31:29           NaN                

In [54]:
print(
    merged[[
        "exercise_duration",
        "exercise_duration_30min",
        "exercise_duration_60min"
    ]].describe()
)

       exercise_duration  exercise_duration_30min  exercise_duration_60min
count       29728.000000             65535.000000             65535.000000
mean           83.358719               226.837079               453.573083
std            48.561494               316.764194               632.966223
min            23.000000                 0.000000                 0.000000
25%            49.000000                 0.000000                 0.000000
50%            70.000000                 0.000000                 0.000000
75%            87.000000               366.000000               732.000000
max           301.000000              1806.000000              3612.000000


In [55]:
print(merged["is_sleeping"].value_counts(dropna=False))

is_sleeping
0    54654
1    10881
Name: count, dtype: int64


In [56]:
print(merged["sleep_quality"].value_counts(dropna=False))

sleep_quality
NaN    41554
3.0    13769
2.0     7947
1.0     2265
Name: count, dtype: int64


In [57]:
merged["sleep_quality"] = pd.to_numeric(
    merged["sleep_quality"],
    errors="coerce"
)

In [58]:
merged["sleep_quality_available"] = (
    merged["sleep_quality"].notna()
).astype(int)

In [59]:
merged["sleep_change"] = (
    merged.groupby("Patient")["is_sleeping"]
    .diff()
)

In [60]:
print(
    merged[[
        "Patient",
        "timestamp",
        "is_sleeping",
        "sleep_change"
    ]].tail(30)
)

               Patient           timestamp  is_sleeping  sleep_change
65505  596-ws-training 2027-05-26 21:34:00            1           0.0
65506  596-ws-training 2027-05-26 21:39:00            1           0.0
65507  596-ws-training 2027-05-26 21:44:00            1           0.0
65508  596-ws-training 2027-05-26 21:49:00            0          -1.0
65509  596-ws-training 2027-05-26 21:54:00            0           0.0
65510  596-ws-training 2027-05-26 21:59:00            0           0.0
65511  596-ws-training 2027-05-26 22:04:00            0           0.0
65512  596-ws-training 2027-05-26 22:09:00            0           0.0
65513  596-ws-training 2027-05-26 22:14:00            0           0.0
65514  596-ws-training 2027-05-26 22:19:00            0           0.0
65515  596-ws-training 2027-05-26 22:24:00            0           0.0
65516  596-ws-training 2027-05-26 22:29:00            0           0.0
65517  596-ws-training 2027-05-26 22:34:00            0           0.0
65518  596-ws-traini

In [61]:
print(
    merged[
        ["gsr", "skin_temperature", "acceleration"]
    ].isna().sum()
)

gsr                 706
skin_temperature    706
acceleration        706
dtype: int64


In [62]:
print(
    merged[
        ["gsr", "skin_temperature", "acceleration"]
    ].describe()
)

                gsr  skin_temperature  acceleration
count  64829.000000      64829.000000  64829.000000
mean       0.742925         52.842833      0.742165
std        3.350052         42.473821      1.503359
min        0.000000          0.000000      0.000000
25%        0.000000          0.000000      0.000000
50%        0.024700         83.690000      0.945465
75%        0.258532         87.810000      1.024543
max       75.074354         96.540000     16.047001


In [63]:
merged["gsr_rolling_15min"] = (
    merged.groupby("Patient")["gsr"]
    .transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
)

In [64]:
merged["skin_temp_rolling_15min"] = (
    merged.groupby("Patient")["skin_temperature"]
    .transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
)

In [65]:
merged["acceleration_rolling_15min"] = (
    merged.groupby("Patient")["acceleration"]
    .transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
)

In [66]:
merged["gsr_rolling_std_30min"] = (
    merged.groupby("Patient")["gsr"]
    .transform(
        lambda x: x.rolling(6, min_periods=2).std()
    )
)

In [67]:
merged["skin_temp_rolling_std_30min"] = (
    merged.groupby("Patient")["skin_temperature"]
    .transform(
        lambda x: x.rolling(6, min_periods=2).std()
    )
)

In [68]:
merged["acceleration_rolling_std_30min"] = (
    merged.groupby("Patient")["acceleration"]
    .transform(
        lambda x: x.rolling(6, min_periods=2).std()
    )
)

In [69]:
print(
    merged[
        [
            "gsr",
            "gsr_rolling_15min",
            "gsr_rolling_std_30min",
            "skin_temperature",
            "skin_temp_rolling_15min",
            "skin_temp_rolling_std_30min",
            "acceleration",
            "acceleration_rolling_15min",
            "acceleration_rolling_std_30min"
        ]
    ].describe()
)

                gsr  gsr_rolling_15min  gsr_rolling_std_30min  \
count  64829.000000       64829.000000           6.482300e+04   
mean       0.742925           0.742928           2.818119e-01   
std        3.350052           3.214252           1.456670e+00   
min        0.000000           0.000000           0.000000e+00   
25%        0.000000           0.000000           3.093724e-07   
50%        0.024700           0.028065           8.367779e-03   
75%        0.258532           0.271690           5.653015e-02   
max       75.074354          75.067283           3.899998e+01   

       skin_temperature  skin_temp_rolling_15min  skin_temp_rolling_std_30min  \
count      64829.000000             64829.000000                 64823.000000   
mean          52.842833                52.845249                     3.719290   
std           42.473821                41.691297                    11.047351   
min            0.000000                 0.000000                     0.000000   
25%      

In [70]:
merged["stress_event"] = (
    merged["stressor_type"].notna()
).astype(int)

In [71]:
merged["illness_event"] = (
    merged["illness_type"].notna()
).astype(int)

In [72]:
merged["hypo_event"] = pd.to_numeric(
    merged["hypo_event"],
    errors="coerce"
).fillna(0).astype(int)

In [73]:
print("Stress events:")
print(merged["stress_event"].value_counts())

print("\nIllness events:")
print(merged["illness_event"].value_counts())

print("\nHypoglycemia events:")
print(merged["hypo_event"].value_counts())

Stress events:
stress_event
0    46756
1    18779
Name: count, dtype: int64

Illness events:
illness_event
0    65535
Name: count, dtype: int64

Hypoglycemia events:
hypo_event
0    37225
1    28310
Name: count, dtype: int64


In [74]:
merged["health_event"] = (
    (merged["stress_event"] == 1) |
    (merged["illness_event"] == 1) |
    (merged["hypo_event"] == 1)
).astype(int)

In [75]:
print(merged["health_event"].value_counts())

health_event
0    37225
1    28310
Name: count, dtype: int64


In [76]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'glucose_prev_5min', 'glucose_prev_10min', 'glucose_prev_15min', 'glucose_prev_30min', 'glucose_change_5min', 'glucose_change_10min', 'glucose_change_15min', 'glucose_change_30min', 'glucose_rate_15min', 'glucose_rate_30min', 'glucose_rate_5min', 'glucose_rolling_mean_15min', 'glucose_rolling_mean_30min', 'glucose_rolling_std_30min', 'glucose_min_30min', 'glucose_max_30min', 'bolus_given', 'bolus_dose_filled', 'bolus_30min', 'bolus_60min', 'basal_rolling_30min', 'basal_rolling_60min', 'temp_basal_active', 'eff

In [77]:
merged.head()

,Patient,timestamp,glucose_value,meal_type,meal_carbs,bolus_type,bolus_dose,basal_value,temp_basal_value,exercise_type,...,sleep_quality_available,sleep_change,gsr_rolling_15min,skin_temp_rolling_15min,acceleration_rolling_15min,gsr_rolling_std_30min,skin_temp_rolling_std_30min,acceleration_rolling_std_30min,stress_event,health_event
0,540-ws-training,2027-05-19 11:36:29,76,NaN,NaN,normal,0.8,0.95,NaN,NaN,...,0,NaN,0.327058,87.170000,0.916837,NaN,NaN,NaN,0,0
1,540-ws-training,2027-05-19 11:41:29,72,NaN,NaN,normal,0.8,0.95,NaN,NaN,...,0,0.0,0.332136,86.820000,0.927254,0.007181,0.494975,0.014732,0,0
2,540-ws-training,2027-05-19 11:46:29,68,NaN,NaN,normal,0.8,0.95,NaN,NaN,...,0,0.0,0.327314,86.676667,0.941179,0.009774,0.429108,0.026272,0,0
3,540-ws-training,2027-05-19 11:51:29,65,NaN,NaN,normal,0.8,0.95,NaN,NaN,...,0,0.0,0.332569,86.666667,0.958709,0.011127,0.420030,0.025684,0,0
4,540-ws-training,2027-05-19 11:56:29,63,NaN,NaN,normal,0.8,0.95,NaN,NaN,...,0,0.0,0.340144,87.043333,0.952951,0.016067,0.512572,0.025491,0,0


In [78]:
missing = merged.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

illness_type                      65535
work_intensity                    60817
illness_description               47434
stressor_description              46756
stressor_type                     46756
sleep_quality                     41554
exercise_type                     35807
exercise_duration                 35807
exercise_intensity                35807
temp_basal_value                   6926
meal_type                          1509
meal_carbs                         1509
skin_temp_rolling_std_30min         712
gsr_rolling_std_30min               712
acceleration_rolling_std_30min      712
acceleration                        706
gsr                                 706
acceleration_rolling_15min          706
skin_temp_rolling_15min             706
gsr_rolling_15min                   706
skin_temperature                    706
bolus_type                          241
bolus_dose                          241
fingerstick_glucose                 179
glucose_rate_30min                   36


In [79]:
missing_pct = (
    merged.isna().mean() * 100
).sort_values(ascending=False)

print(
    missing_pct[missing_pct > 0]
)

illness_type                      100.000000
work_intensity                     92.800793
illness_description                72.379644
stressor_description               71.345083
stressor_type                      71.345083
sleep_quality                      63.407340
exercise_type                      54.637980
exercise_duration                  54.637980
exercise_intensity                 54.637980
temp_basal_value                   10.568399
meal_carbs                          2.302586
meal_type                           2.302586
acceleration_rolling_std_30min      1.086442
gsr_rolling_std_30min               1.086442
skin_temp_rolling_std_30min         1.086442
gsr                                 1.077287
skin_temperature                    1.077287
acceleration                        1.077287
skin_temp_rolling_15min             1.077287
gsr_rolling_15min                   1.077287
acceleration_rolling_15min          1.077287
bolus_dose                          0.367742
bolus_type

In [80]:
missing = merged.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

illness_type                      65535
work_intensity                    60817
illness_description               47434
stressor_description              46756
stressor_type                     46756
sleep_quality                     41554
exercise_type                     35807
exercise_duration                 35807
exercise_intensity                35807
temp_basal_value                   6926
meal_type                          1509
meal_carbs                         1509
skin_temp_rolling_std_30min         712
gsr_rolling_std_30min               712
acceleration_rolling_std_30min      712
acceleration                        706
gsr                                 706
acceleration_rolling_15min          706
skin_temp_rolling_15min             706
gsr_rolling_15min                   706
skin_temperature                    706
bolus_type                          241
bolus_dose                          241
fingerstick_glucose                 179
glucose_rate_30min                   36


In [81]:
missing_pct = (merged.isna().mean() * 100).sort_values(ascending=False)

print(missing_pct[missing_pct > 0])

illness_type                      100.000000
work_intensity                     92.800793
illness_description                72.379644
stressor_description               71.345083
stressor_type                      71.345083
sleep_quality                      63.407340
exercise_type                      54.637980
exercise_duration                  54.637980
exercise_intensity                 54.637980
temp_basal_value                   10.568399
meal_carbs                          2.302586
meal_type                           2.302586
acceleration_rolling_std_30min      1.086442
gsr_rolling_std_30min               1.086442
skin_temp_rolling_std_30min         1.086442
gsr                                 1.077287
skin_temperature                    1.077287
acceleration                        1.077287
skin_temp_rolling_15min             1.077287
gsr_rolling_15min                   1.077287
acceleration_rolling_15min          1.077287
bolus_dose                          0.367742
bolus_type

In [82]:
event_cols = [
    "meal_event",
    "bolus_given",
    "temp_basal_active",
    "exercise_event",
    "stress_event",
    "illness_event",
    "hypo_event",
    "health_event"
]

merged[event_cols] = merged[event_cols].fillna(0)

In [83]:
merged["meal_carbs"] = merged["meal_carbs"].fillna(0)

In [84]:
merged["bolus_dose_filled"] = merged["bolus_dose_filled"].fillna(0)

In [85]:
merged["exercise_duration"] = merged["exercise_duration"].fillna(0)

In [86]:
# Make a copy before missing-value handling
df = merged.copy()

# -----------------------------
# 1. Event/category columns
# -----------------------------

categorical_cols = [
    "meal_type",
    "bolus_type",
    "exercise_type",
    "exercise_intensity",
    "stressor_type",
    "stressor_description",
    "illness_type",
    "illness_description",
    "work_intensity"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("none")


# -----------------------------
# 2. Event indicator columns
# -----------------------------

event_cols = [
    "bolus_given",
    "meal_event",
    "exercise_event",
    "sleep_quality_available",
    "stress_event",
    "illness_event",
    "health_event"
]

for col in event_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)


# -----------------------------
# 3. Event quantity columns
# -----------------------------

zero_cols = [
    "meal_carbs",
    "bolus_dose_filled",
    "exercise_duration",
    "exercise_duration_30min",
    "exercise_duration_60min",
    "carbs_30min",
    "carbs_60min"
]

for col in zero_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)


# -----------------------------
# 4. Temp basal
# -----------------------------

if "temp_basal_value" in df.columns:
    df["temp_basal_value"] = df["temp_basal_value"].fillna(0)


# -----------------------------
# 5. Check remaining missing values
# -----------------------------

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Remaining missing values:")
print(missing)

Remaining missing values:
sleep_quality                     41554
acceleration_rolling_std_30min      712
skin_temp_rolling_std_30min         712
gsr_rolling_std_30min               712
gsr                                 706
skin_temperature                    706
acceleration                        706
acceleration_rolling_15min          706
skin_temp_rolling_15min             706
gsr_rolling_15min                   706
bolus_dose                          241
fingerstick_glucose                 179
glucose_change_30min                 36
glucose_rate_30min                   36
glucose_prev_30min                   36
glucose_rate_15min                   18
glucose_prev_15min                   18
glucose_change_15min                 18
glucose_change_10min                 12
glucose_prev_10min                   12
glucose_change_5min                   6
glucose_rate_5min                     6
glucose_rolling_std_30min             6
sleep_change                          6
glucose_prev_5

In [87]:
print(df[[
    "bolus_dose",
    "bolus_dose_filled",
    "bolus_30min",
    "bolus_60min",
    "exercise_duration",
    "exercise_duration_30min",
    "exercise_duration_60min"
]].describe())

         bolus_dose  bolus_dose_filled   bolus_30min   bolus_60min  \
count  65294.000000       65535.000000  65535.000000  65535.000000   
mean       6.792060           6.767082     40.592538     81.162515   
std        4.854745           4.863219     28.774137     56.727741   
min        0.100000           0.000000      0.000000      0.000000   
25%        3.000000           2.900000     18.000000     36.000000   
50%        5.900000           5.800000     35.400000     70.800000   
75%        9.300000           9.300000     55.800000    111.600000   
max       25.000000          25.000000    150.000000    300.000000   

       exercise_duration  exercise_duration_30min  exercise_duration_60min  
count       65535.000000             65535.000000             65535.000000  
mean           37.813199               226.837079               453.573083  
std            52.839040               316.764194               632.966223  
min             0.000000                 0.000000            

In [88]:
# Remove incorrectly calculated rolling event features
df = df.drop(columns=[
    "bolus_30min",
    "bolus_60min",
    "exercise_duration_30min",
    "exercise_duration_60min"
], errors="ignore")

print(df.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'glucose_prev_5min', 'glucose_prev_10min', 'glucose_prev_15min', 'glucose_prev_30min', 'glucose_change_5min', 'glucose_change_10min', 'glucose_change_15min', 'glucose_change_30min', 'glucose_rate_15min', 'glucose_rate_30min', 'glucose_rate_5min', 'glucose_rolling_mean_15min', 'glucose_rolling_mean_30min', 'glucose_rolling_std_30min', 'glucose_min_30min', 'glucose_max_30min', 'bolus_given', 'bolus_dose_filled', 'basal_rolling_30min', 'basal_rolling_60min', 'temp_basal_active', 'effective_basal', 'basal_change',

In [89]:
print(df[[
    "meal_carbs",
    "carbs_30min",
    "carbs_60min"
]].describe())

         meal_carbs   carbs_30min   carbs_60min
count  65535.000000  65535.000000  65535.000000
mean      54.560586    327.298054    654.439002
std       31.269054    186.376428    370.187733
min        0.000000      0.000000      0.000000
25%       30.000000    180.000000    360.000000
50%       60.000000    360.000000    720.000000
75%       73.000000    434.500000    864.000000
max      162.000000    972.000000   1944.000000


In [90]:
print(df[[
    "bolus_given",
    "bolus_dose_filled",
    "exercise_event",
    "exercise_duration"
]].describe())

        bolus_given  bolus_dose_filled  exercise_event  exercise_duration
count  65535.000000       65535.000000    65535.000000       65535.000000
mean       0.996323           6.767082        0.453620          37.813199
std        0.060531           4.863219        0.497848          52.839040
min        0.000000           0.000000        0.000000           0.000000
25%        1.000000           2.900000        0.000000           0.000000
50%        1.000000           5.800000        0.000000           0.000000
75%        1.000000           9.300000        1.000000          61.000000
max        1.000000          25.000000        1.000000         301.000000


In [91]:
df = df.drop(columns=[
    "bolus_30min",
    "bolus_60min",
    "carbs_30min",
    "carbs_60min",
    "exercise_duration_30min",
    "exercise_duration_60min"
], errors="ignore")

print("Updated shape:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

Updated shape: (65535, 65)

Remaining columns:
['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'glucose_prev_5min', 'glucose_prev_10min', 'glucose_prev_15min', 'glucose_prev_30min', 'glucose_change_5min', 'glucose_change_10min', 'glucose_change_15min', 'glucose_change_30min', 'glucose_rate_15min', 'glucose_rate_30min', 'glucose_rate_5min', 'glucose_rolling_mean_15min', 'glucose_rolling_mean_30min', 'glucose_rolling_std_30min', 'glucose_min_30min', 'glucose_max_30min', 'bolus_given', 'bolus_dose_filled', 'basal_rolling_30min', 'basal_rolling_60min', 'temp_ba

In [92]:
df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

print(df[["Patient", "timestamp", "glucose_value"]].head(10))

           Patient           timestamp  glucose_value
0  540-ws-training 2027-05-19 11:36:29             76
1  540-ws-training 2027-05-19 11:41:29             72
2  540-ws-training 2027-05-19 11:46:29             68
3  540-ws-training 2027-05-19 11:51:29             65
4  540-ws-training 2027-05-19 11:56:29             63
5  540-ws-training 2027-05-19 12:01:29             66
6  540-ws-training 2027-05-19 12:06:29             71
7  540-ws-training 2027-05-19 12:11:29             78
8  540-ws-training 2027-05-19 12:16:29             90
9  540-ws-training 2027-05-19 12:21:29             99


In [93]:
time_check = (
    df.groupby("Patient")["timestamp"]
      .diff()
      .dt.total_seconds()
)

print("Negative time differences:", (time_check < 0).sum())
print("Zero time differences:", (time_check == 0).sum())

Negative time differences: 0
Zero time differences: 0


In [94]:
df.groupby("Patient")["timestamp"].diff().dt.total_seconds().describe()

count     65529.000000
mean        353.822506
std        2013.636703
min         179.000000
25%         300.000000
50%         300.000000
75%         300.000000
max      221700.000000
Name: timestamp, dtype: float64

In [95]:
# Remove columns from the failed attempt
df = df.drop(
    columns=["target_timestamp", "future_timestamp", "glucose_30min_ahead"],
    errors="ignore"
)

In [96]:
import pandas as pd

# Make sure data is correctly sorted
df = df.sort_values(["Patient", "timestamp"]).reset_index(drop=True)

# Create target timestamp
df["target_timestamp"] = df["timestamp"] + pd.Timedelta(minutes=30)

# Create empty target column
df["glucose_30min_ahead"] = pd.NA

# Process each patient separately
for patient in df["Patient"].unique():

    patient_mask = df["Patient"] == patient

    current_times = df.loc[patient_mask, "target_timestamp"]
    
    future_data = df.loc[
        patient_mask,
        ["timestamp", "glucose_value"]
    ].sort_values("timestamp")

    # Find the first glucose reading at or after the target time
    matched = pd.merge_asof(
        current_times.to_frame(),
        future_data,
        left_on="target_timestamp",
        right_on="timestamp",
        direction="forward",
        tolerance=pd.Timedelta(minutes=2)
    )

    df.loc[
        patient_mask,
        "glucose_30min_ahead"
    ] = matched["glucose_value"].values

In [97]:
print(
    df[[
        "Patient",
        "timestamp",
        "target_timestamp",
        "glucose_30min_ahead"
    ]].head(20)
)

            Patient           timestamp    target_timestamp  \
0   540-ws-training 2027-05-19 11:36:29 2027-05-19 12:06:29   
1   540-ws-training 2027-05-19 11:41:29 2027-05-19 12:11:29   
2   540-ws-training 2027-05-19 11:46:29 2027-05-19 12:16:29   
3   540-ws-training 2027-05-19 11:51:29 2027-05-19 12:21:29   
4   540-ws-training 2027-05-19 11:56:29 2027-05-19 12:26:29   
5   540-ws-training 2027-05-19 12:01:29 2027-05-19 12:31:29   
6   540-ws-training 2027-05-19 12:06:29 2027-05-19 12:36:29   
7   540-ws-training 2027-05-19 12:11:29 2027-05-19 12:41:29   
8   540-ws-training 2027-05-19 12:16:29 2027-05-19 12:46:29   
9   540-ws-training 2027-05-19 12:21:29 2027-05-19 12:51:29   
10  540-ws-training 2027-05-19 12:26:29 2027-05-19 12:56:29   
11  540-ws-training 2027-05-19 12:31:29 2027-05-19 13:01:29   
12  540-ws-training 2027-05-19 12:36:29 2027-05-19 13:06:29   
13  540-ws-training 2027-05-19 12:41:29 2027-05-19 13:11:29   
14  540-ws-training 2027-05-19 12:46:29 2027-05-19 13:1

In [98]:
print("\nMissing target values:")
print(df["glucose_30min_ahead"].isna().sum())

print("\nTarget statistics:")
print(
    pd.to_numeric(df["glucose_30min_ahead"], errors="coerce")
      .describe()
)



Missing target values:
1252

Target statistics:
count    64283.000000
mean       157.633822
std         60.818517
min         40.000000
25%        112.000000
50%        149.000000
75%        193.000000
max        400.000000
Name: glucose_30min_ahead, dtype: float64


In [99]:
print("\nMissing target values:")
print(df["glucose_30min_ahead"].isna().sum())

print("\nTarget statistics:")
print(
    pd.to_numeric(df["glucose_30min_ahead"], errors="coerce")
      .describe()
)


Missing target values:
1252

Target statistics:
count    64283.000000
mean       157.633822
std         60.818517
min         40.000000
25%        112.000000
50%        149.000000
75%        193.000000
max        400.000000
Name: glucose_30min_ahead, dtype: float64


In [100]:
before = len(df)

df = df.dropna(
    subset=["glucose_30min_ahead"]
).reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 65535
Rows after: 64283
Rows removed: 1252


In [101]:
df = df.drop(
    columns=[
        "target_timestamp",
        "future_timestamp"
    ],
    errors="ignore"
)

In [102]:
target = "glucose_30min_ahead"

X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget:")
print(y.describe())

X shape: (64283, 65)
y shape: (64283,)

Target:
count     64283.0
unique      359.0
top         130.0
freq        474.0
Name: glucose_30min_ahead, dtype: float64


In [103]:
print("Final dataset shape:", df.shape)

print("\nTarget missing:")
print(df["glucose_30min_ahead"].isna().sum())

print("\nPatients:")
print(df["Patient"].nunique())

print("\nColumns:")
print(df.columns.tolist())

Final dataset shape: (64283, 66)

Target missing:
0

Patients:
6

Columns:
['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'glucose_prev_5min', 'glucose_prev_10min', 'glucose_prev_15min', 'glucose_prev_30min', 'glucose_change_5min', 'glucose_change_10min', 'glucose_change_15min', 'glucose_change_30min', 'glucose_rate_15min', 'glucose_rate_30min', 'glucose_rate_5min', 'glucose_rolling_mean_15min', 'glucose_rolling_mean_30min', 'glucose_rolling_std_30min', 'glucose_min_30min', 'glucose_max_30min', 'bolus_given', 'bolus_dose_filled', 'basal_rolling_30min', 'ba

In [104]:
df.to_csv(
    "../data/processed/feature_engineered.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.
